# Speech Emotion Recognition — Quickstart

End-to-end walkthrough: load data → extract MFCC features → train a CNN-LSTM → evaluate → predict on a new file.

Run this notebook from the project root (`speech-emotion-recognition/`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')  # run from project root

from src.utils import load_config, set_seed, EMOTION_LABELS, NUM_CLASSES
cfg = load_config('config.yaml')
set_seed(cfg['dataset']['random_seed'])
print('Emotion classes:', EMOTION_LABELS)

## 1. Load dataset & inspect class balance

Make sure you've downloaded RAVDESS (or TESS/EMO-DB) into `data/RAVDESS/` first (see README.md).

In [ ]:
from src.dataset_loader import load_dataset, dataset_summary

DATASET = 'ravdess'
DATA_DIR = 'data/RAVDESS'

triples = load_dataset(DATASET, DATA_DIR)
_ = dataset_summary(triples)

## 2. Listen to a sample & visualize its waveform + MFCCs

In [ ]:
import librosa, librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd

sample_path, sample_label, _ = triples[0]
print(f'Sample: {sample_path}  |  Emotion: {sample_label}')

y, sr = librosa.load(sample_path, sr=cfg['audio']['sample_rate'])
ipd.display(ipd.Audio(y, rate=sr))

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
librosa.display.waveshow(y, sr=sr, ax=axes[0])
axes[0].set_title(f'Waveform — {sample_label}')

mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=cfg['features']['n_mfcc'])
img = librosa.display.specshow(mfcc, x_axis='time', ax=axes[1])
axes[1].set_title('MFCCs')
fig.colorbar(img, ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Extract features for the full dataset (cached to disk)

In [ ]:
from src.feature_extraction import build_feature_dataset
import numpy as np, os

feat_path = os.path.join(cfg['dataset']['processed_dir'], f'{DATASET}_features.npz')

if os.path.exists(feat_path):
    data = np.load(feat_path, allow_pickle=True)
    X, y, label_names = data['X'], data['y'], data['label_names']
    print(f'Loaded cached features: {X.shape}')
else:
    X, y, label_names = build_feature_dataset(triples, cfg)
    os.makedirs(cfg['dataset']['processed_dir'], exist_ok=True)
    np.savez_compressed(feat_path, X=X, y=y, label_names=label_names)
    print(f'Extracted and cached features: {X.shape}')

## 4. Train a CNN-LSTM model

For a full run, it's usually faster to use the CLI:
```bash
python src/train.py --model cnn_lstm --epochs 50
```
But here's the equivalent inline, using a few epochs for demonstration.

In [ ]:
!python src/train.py --dataset {DATASET} --model cnn_lstm --epochs 15

## 5. Evaluate on the held-out test set

In [ ]:
!python src/evaluate.py --model_path outputs/best_model.pt

In [ ]:
from PIL import Image
Image.open('outputs/confusion_matrix.png')

## 6. Predict on a new audio file

In [ ]:
test_file = triples[10][0]  # swap in any .wav path you like
!python src/predict.py --file "{test_file}" --model_path outputs/best_model.pt

## Next steps
- Try `--model cnn` or `--model lstm` and compare test accuracy.
- Re-run feature extraction with `--augment` to add noise/pitch/time-stretch augmentation.
- Launch the interactive demo: `streamlit run app.py`.